In [0]:
# Create sample data
data = [
 (1, "Navneet", 85),
 (2, "Harsh", 90),
 (3, "Ankit", 78)
]
columns = ["student_id", "name", "score"]
df = spark.createDataFrame(data, columns)
df.show()


+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|Navneet|   85|
|         2|  Harsh|   90|
|         3|  Ankit|   78|
+----------+-------+-----+



In [0]:
#CREATE DELTA TABLE
spark.sql("CREATE TABLE IF NOT EXISTS students (student_id LONG, name STRING, score LONG) USING DELTA")

DataFrame[]

In [0]:
# Save as Delta table
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.students")
# Read back
delta_df = spark.table("workspace.default.students")
delta_df.show()


+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|Navneet|   85|
|         2|  Harsh|   90|
|         3|  Ankit|   78|
+----------+-------+-----+



In [0]:
from delta.tables import DeltaTable

# Load Delta table
delta_table = DeltaTable.forName(spark, "workspace.default.students")
# Update score for Navneet
delta_table.update(
 condition = "name = 'Navneet'",
 set = { "score": "95" }
)
# Show updated data
delta_table.toDF().show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|Navneet|   95|
|         2|  Harsh|   90|
|         3|  Ankit|   78|
+----------+-------+-----+



In [0]:
new_data = [(4, "David", 88)]
new_df = spark.createDataFrame(new_data, columns)
new_df.write.format("delta").mode("append").saveAsTable("workspace.default.students")
# Show current table
spark.table("workspace.default.students").show()


+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         2|  Harsh|   90|
|         3|  Ankit|   78|
|         1|Navneet|   95|
|         4|  David|   88|
+----------+-------+-----+



In [0]:
# Check history
delta_table.history().show()

+-------+-------------------+--------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      7|2026-10-09 14:15:21|74473584144393|navneetmestry010@...|               WRITE|{mode -> Append, ...|NULL|{4230991577834743}|   1f06ee9f-18e5-4ca...|1009-134036-g1pqr

In [0]:

# Read version 1 (original data)
spark.read.format("delta").option("versionAsOf", 1).table("workspace.default.students").show()
# Read version 3 (after Alice update)
spark.read.format("delta").option("versionAsOf", 3).table("workspace.default.students").show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|Navneet|   85|
|         2|  Harsh|   90|
|         3|  Ankit|   78|
+----------+-------+-----+

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         2|  Harsh|   90|
|         3|  Ankit|   78|
|         1|Navneet|   95|
+----------+-------+-----+

